#**Assignment 3 - Group 13**

---

## Context: Newsletter with different typography, line-spacing and viewport sizes.
Purpose: To demonstrate A) Types of randomized control trials, B) Subsetting data and C) Hypothesis testing

By Janna Cameron, YiFan Dai, Cici Liu and Margot Whitfield



In Assignment 3, a blocking factor (Reading Difficulty: easy, medium, hard) was introduced to control for variation in text complexity across participants. This allows for more precise estimation of treatment effects by reducing unexplained variability.

# Package Installation

In [ ]:
# Install statistics packages
!pip install pingouin
import pingouin as pg

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.0/204.0 kB 4.5 MB/s eta 0:00:00


In [ ]:
# Install flowchart package
!pip install pyflowchart
from pyflowchart import Flowchart

In [ ]:
# Import necessary libraries for statistic model generation
import statsmodels.api as sm
import statsmodels.stats.multicomp as mc
import statsmodels.formula.api as smf
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm
from scipy import stats
from scipy.stats import chi2
from scipy.stats import chisquare
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# Import necessary libraries for synthetic data generation and logic
import pandas as pd
import numpy as np
from math import sqrt
import math
import itertools

# Import necessary libraries for power analsyis
from statsmodels.stats.power import FTestAnovaPower
from statsmodels.stats.power import TTestPower

# Import for plotting
import matplotlib.pyplot as plt
import seaborn as sns

# Synthetic Data Generation

In [ ]:
def generate_trial_data(n_participants):
    np.random.seed(42)

    # --- BASELINE VARIABLES (For Parallel, Factorial, & Matched Pairs) ---
    # These stay randomized to allow you to subset by 'Screen_Size' or 'Line_Spacing' later.
    screens = ['small', 'medium', 'large']
    fonts = ['arial', 'helvetica', 'courier']
    spacings = ['single', 'one_and_half', 'one_three_quarters']

    all_combos = list(itertools.product(screens, fonts, spacings))
    repeats = (n_participants // len(all_combos)) + 1
    assignments = (all_combos * repeats)[:n_participants]

    data = []
    weeks_list = [f'Week_{i}' for i in range(1, 9)]

    for p_id, (s_assigned, f_assigned, sp_assigned) in enumerate(assignments):
        p_age = np.random.randint(18, 75)

        # --- GROUP ASSIGNMENT LOGIC (For Crossover & Withdrawal) ---
        # Withdrawal treatment and switch to control
        withdraw_grp = 'Withdraw_to_Control' if p_id % 2 == 0 else 'Stay_on_Treatment'
        # Crossover Sequence of fonts (Helvetica-Courier vs Courier-Helvetica)
        sequence = 'CH' if p_id % 2 == 0 else 'HC'

        for week in weeks_list:
            week_num = int(week.split('_')[1])
            base_time = 150 + (p_age * 0.2) # Age effect for Matched Pairs

            # --- CROSSOVER PERIODS & WASHOUT LOGIC ---
            # Period and Typeface columns
            if week_num <= 3:
                period = 'Period_1'
                current_font = 'courier' if sequence == 'CH' else 'helvetica'
            elif week_num in [4, 5]:
                period = 'Washout'
                current_font = 'Standard'
            else:
                period = 'Period_2'
                current_font = 'helvetica' if sequence == 'CH' else 'courier'

            # --- WITHDRAWAL SPACING LOGIC ---
            # This handles the 1.75 to 1.0 spacing shift for Withdrawal RCT
            if week_num <= 4:
                # Everyone on 1.75
                current_spacing = 'one_three_quarters'
                base_time -= 5
            else:
                # Divergence
                if withdraw_grp == 'Withdraw_to_Control':
                    current_spacing = 'single'
                    base_time += 10 # Slow down effect
                else:
                    current_spacing = 'one_three_quarters'
                    base_time -= 5

            # --- STATIC LOGIC (Parallel & Factorial) ---
            # Ensures Screen_Size still has a predictable impact on results
            if s_assigned == 'large': base_time -= 10
            # if s_assigned == 'small': base_time += 10

            noise = np.random.normal(0, 3)
            data.append({
                'Participant_ID': p_id,
                'Age': p_age,
                'Screen_Size': s_assigned,      # Used for Parallel/Factorial
                'Line_Spacing': sp_assigned,    # Used for Matched Pairs
                'Withdrawal_Group': withdraw_grp, # Used for Withdrawal ANOVA
                'Typeface': current_font,       # Used for Crossover
                'Time_Point': week,
                'Week_Number': week_num,
                'Period': period,               # Used for Crossover
                'Actual_Spacing': current_spacing, # Used for Withdrawal
                'Reading_Time': round(base_time + noise, 2)
            })

    return pd.DataFrame(data)


# Generate raw dataset
df_raw= generate_trial_data(300)

# df_raw['Reading_Difficulty'] = np.random.choice(
#     ['Low', 'Medium', 'High'],
#     size=len(df_raw)
# )

In [ ]:
df = generate_trial_data(5)
df

,Participant_ID,Age,Screen_Size,Line_Spacing,Withdrawal_Group,Typeface,Time_Point,Week_Number,Period,Actual_Spacing,Reading_Time
0,0,56,small,single,Withdraw_to_Control,courier,Week_1,1,Period_1,one_three_quarters,154.55
1,0,56,small,single,Withdraw_to_Control,courier,Week_2,2,Period_1,one_three_quarters,157.75
2,0,56,small,single,Withdraw_to_Control,courier,Week_3,3,Period_1,one_three_quarters,157.62
3,0,56,small,single,Withdraw_to_Control,Standard,Week_4,4,Washout,one_three_quarters,160.31
4,0,56,small,single,Withdraw_to_Control,Standard,Week_5,5,Washout,single,168.45
5,0,56,small,single,Withdraw_to_Control,helvetica,Week_6,6,Period_2,single,170.83
6,0,56,small,single,Withdraw_to_Control,helvetica,Week_7,7,Period_2,single,165.17
7,0,56,small,single,Withdraw_to_Control,helvetica,Week_8,8,Period_2,single,169.72
8,1,57,small,one_and_half,Stay_on_Treatment,helvetica,Week_1,1,Period_1,one_three_quarters,154.66
9,1,57,small,one_and_half,Stay_on_Treatment,helvetica,Week_2,2,Period_1,one_three_quarters,154.82



## Type 1: Parallel RCT

Our objective is to understand the effect of screen size on reading time. This is a between-subjects test. Assess whether screen size is associated with differences in reading time (Reading_Time) among independent groups in week 8.

**H_0:** There is no difference in the mean reading time across the different screen size groups.

**H_A:** At least one screen size group has a significantly different mean reading time compared to the others.

**Factor:** Screen Size {Small, Medium, Large}

**Treatments:** Medium screen, large screen

**Control:** Small screen

**Outcome Variable:** Reading_Time

**Other features:** Independent groups; participants are randomly assigned to one screen size condition and measured once in week 8. Randomized complete block design with blocks defined by reading difficulty (Easy, Medium, Hard).



**Subset Data**

In [ ]:
# Create the pilot dataset using your synthetic generator
parallel_pilot_raw = generate_trial_data(159)

# Group by Screen_Size and restrict to Week_8
df_parallel_pilot = parallel_pilot_raw[parallel_pilot_raw['Time_Point'] == 'Week_8'].copy()

# Preview the subset to ensure Week_8 data is captured
print("Pilot Subset Data (Week 8):")
print(df_parallel_pilot[['Participant_ID', 'Screen_Size', 'Reading_Time']].head())

Pilot Subset Data (Week 8):
    Participant_ID Screen_Size  Reading_Time
7                0       small        169.72
15               1       small        151.83
23               2       small        174.30
31               3       small        146.54
39               4       small        163.29


**Blocking Factor**




In [ ]:
# Add blocking factor to the parallel subset only
np.random.seed(42)
df_parallel_pilot['Reading_Difficulty'] = np.random.choice(
    ['easy', 'medium', 'hard'],
    size=len(df_parallel_pilot)
)

print("Pilot Subset Data with Blocking Factor:")
print(df_parallel_pilot[['Participant_ID', 'Screen_Size', 'Reading_Difficulty', 'Reading_Time']].head())

Pilot Subset Data with Blocking Factor:
    Participant_ID Screen_Size Reading_Difficulty  Reading_Time
7                0       small               hard        169.72
15               1       small               easy        151.83
23               2       small               hard        174.30
31               3       small               hard        146.54
39               4       small               easy        163.29


**Model & Results One-Way ANOVA (Parallel RCT)**



In [ ]:
# Fit ANOVA with blocking factor
# Treatment factor: Screen_Size
# Blocking factor: Reading_Difficulty
formula = 'Reading_Time ~ C(Screen_Size) + C(Reading_Difficulty)'

model_parallel_block = ols(formula, data=df_parallel_pilot).fit()
anova_results = sm.stats.anova_lm(model_parallel_block, typ=2)

print("--- Parallel RCT ANOVA Results with Blocking Factor ---")
print(anova_results)

--- Parallel RCT ANOVA Results with Blocking Factor ---
                             sum_sq     df          F    PR(>F)
C(Screen_Size)          2038.198437    2.0  14.001294  0.000003
C(Reading_Difficulty)     10.114430    2.0   0.069481  0.932908
Residual               11209.055762  154.0        NaN       NaN


In [ ]:
posthoc = pairwise_tukeyhsd(
    endog=df_parallel_pilot['Reading_Time'],
    groups=df_parallel_pilot['Screen_Size'],
    alpha=0.05
)

print("--- Tukey HSD Post-Hoc Results ---")
print(posthoc)

--- Tukey HSD Post-Hoc Results ---
Multiple Comparison of Means - Tukey HSD, FWER=0.05 
group1 group2 meandiff p-adj   lower   upper  reject
----------------------------------------------------
 large medium   7.8607    0.0  3.9424  11.779   True
 large  small   7.4246    0.0  3.5063 11.3429   True
medium  small  -0.4361 0.9614 -4.2981  3.4258  False
----------------------------------------------------


**Effect Calculation**

In [ ]:
# Rename the subset
df_effect_calc = df_parallel_pilot.copy()

# Calculate means
mean_small  = df_effect_calc[df_effect_calc['Screen_Size'] == 'small']['Reading_Time'].mean()
mean_medium = df_effect_calc[df_effect_calc['Screen_Size'] == 'medium']['Reading_Time'].mean()
mean_large  = df_effect_calc[df_effect_calc['Screen_Size'] == 'large']['Reading_Time'].mean()
ave_time_total = df_effect_calc['Reading_Time'].mean()

# Calculate standard deviation of the group means
sd_screen_effects = sqrt(
    ((mean_small - ave_time_total)**2 +
     (mean_medium - ave_time_total)**2 +
     (mean_large - ave_time_total)**2) / 3
)

# Calculate pooled variance
n_small = len(df_effect_calc[df_effect_calc['Screen_Size'] == 'small'])
n_med   = len(df_effect_calc[df_effect_calc['Screen_Size'] == 'medium'])
n_large = len(df_effect_calc[df_effect_calc['Screen_Size'] == 'large'])

var_small = df_effect_calc[df_effect_calc['Screen_Size'] == 'small']['Reading_Time'].var()
var_med   = df_effect_calc[df_effect_calc['Screen_Size'] == 'medium']['Reading_Time'].var()
var_large = df_effect_calc[df_effect_calc['Screen_Size'] == 'large']['Reading_Time'].var()

numerator = ((n_small - 1) * var_small +
             (n_med - 1) * var_med +
             (n_large - 1) * var_large)
denominator = (n_small + n_med + n_large) - 3

s_pooled_parallel = sqrt(numerator / denominator)

# Cohen's f
f_parallel = sd_screen_effects / s_pooled_parallel

print(f"Effect Size (f): {f_parallel:.3f}")

Effect Size (f): 0.426


In [ ]:
# Data descriptions

# Calculate sd
sd_small = sqrt(var_small)
sd_med = sqrt(var_med)
sd_large = sqrt(var_large)

print("The mean and sd for small screens is: ", round(mean_small,2), round(sd_small,2) )
print("The mean for medium screens is: ", round(mean_medium,2), round(sd_med,2))
print("The mean for large screens is: ", round(mean_large,2), round(sd_large,2))

The mean and sd for small screens is:  160.5 8.85
The mean for medium screens is:  160.93 8.12
The mean for large screens is:  153.07 8.46


## Type 2: Factorial RCT

Our objective is to understand the effects of screen size and line spacing and the interaction effect between the two factors. This is a between-subjects test.

**Main effect of screen size**

H_01: Screen size has no effect on reading time.

H_A1: Screen size affects reading time.

**Main effect of line spacing**

H_02: Line spacing has no effect on reading time.

H_A2: Line spacing affects reading time.

**Interaction**

H_03: There is no interaction between Screen Size and Line Spacing (the effect of Line Spacing on Reading_Time does not depend on Screen Size).

H_A3: There is an interaction between Screen Size and Line Spacing.

**Factors:**
	Screen size: {Small, Medium, Large}
	Line spacing: {1, 1.5, 1.75}

**Treatments:** Screen size and line spacing

**Interaction:** Screen size and line spacing

**Outcome Variable:** Reading_Time

**Other features:** Independent groups; participants are randomly assigned to one screen size and line spacing conditions and measured once in week 8. Randomized complete block design with blocks defined by reading difficulty (Easy, Medium, Hard).

**Subset Data**

In [ ]:
# Create the pilot dataset using synthetic data
factorial_pilot_raw = generate_trial_data(252)

# Group by Screen_Size and Line_Spacing and restrict to Week_8
df_factorial_pilot = factorial_pilot_raw[factorial_pilot_raw['Time_Point'] == 'Week_8'].copy()

# Preview the subset
print("Pilot Subset Data (Week 8):")
print(df_factorial_pilot[['Participant_ID', 'Screen_Size', 'Line_Spacing', 'Reading_Time']].head())

Pilot Subset Data (Week 8):
    Participant_ID Screen_Size        Line_Spacing  Reading_Time
7                0       small              single        169.72
15               1       small        one_and_half        151.83
23               2       small  one_three_quarters        174.30
31               3       small              single        146.54
39               4       small        one_and_half        163.29


**Adding blocking factor**

In [ ]:
# Add blocking factor to the factorial subset only
np.random.seed(42)
df_factorial_pilot['Reading_Difficulty'] = np.random.choice(
    ['easy', 'medium', 'hard'],
    size=len(df_factorial_pilot)
)

print("Pilot Subset Data with Blocking Factor:")
print(df_factorial_pilot[['Participant_ID', 'Screen_Size', 'Line_Spacing', 'Reading_Difficulty', 'Reading_Time']].head())

Pilot Subset Data with Blocking Factor:
    Participant_ID Screen_Size        Line_Spacing Reading_Difficulty  \
7                0       small              single               hard   
15               1       small        one_and_half               easy   
23               2       small  one_three_quarters               hard   
31               3       small              single               hard   
39               4       small        one_and_half               easy   

    Reading_Time  
7         169.72  
15        151.83  
23        174.30  
31        146.54  
39        163.29  


**Model & Results Two-Way ANOVA (Factorial RCT)**

In [ ]:
# Fit a factorial ANOVA model with interaction and blocking factor
formula = 'Reading_Time ~ C(Screen_Size) * C(Line_Spacing) + C(Reading_Difficulty)'

model_factorial = ols(formula, data=df_factorial_pilot).fit()
anova_factorial_results = sm.stats.anova_lm(model_factorial, typ=2)

print("--- Factorial RCT ANOVA Results with Blocking Factor ---")
print(anova_factorial_results)

--- Factorial RCT ANOVA Results with Blocking Factor ---
                                      sum_sq     df          F        PR(>F)
C(Screen_Size)                   4777.585190    2.0  33.472506  1.486060e-13
C(Line_Spacing)                    18.920860    2.0   0.132562  8.759120e-01
C(Reading_Difficulty)             140.706586    2.0   0.985812  3.746355e-01
C(Screen_Size):C(Line_Spacing)    152.207824    4.0   0.533196  7.114615e-01
Residual                        17199.160989  241.0        NaN           NaN


In [ ]:
# Tukey post hoc for Screen_Size
tukey = pairwise_tukeyhsd(
    endog=df_factorial_pilot['Reading_Time'],
    groups=df_factorial_pilot['Screen_Size'],
    alpha=0.05
)

print("--- Tukey HSD: Screen Size ---")
print(tukey)

--- Tukey HSD: Screen Size ---
Multiple Comparison of Means - Tukey HSD, FWER=0.05 
group1 group2 meandiff p-adj   lower   upper  reject
----------------------------------------------------
 large medium   9.5817    0.0  6.4777 12.6858   True
 large  small   9.0745    0.0  6.0491    12.1   True
medium  small  -0.5072 0.9175 -3.5327  2.5182  False
----------------------------------------------------


**Effect Calculation**

In [ ]:
# Calculate partial eta-squared
ss_screen = anova_factorial_results.loc['C(Screen_Size)', 'sum_sq']
ss_spacing = anova_factorial_results.loc['C(Line_Spacing)', 'sum_sq']
ss_interaction = anova_factorial_results.loc['C(Screen_Size):C(Line_Spacing)', 'sum_sq']
ss_residual = anova_factorial_results.loc['Residual', 'sum_sq']

eta_p_screen = ss_screen / (ss_screen + ss_residual)
eta_p_spacing = ss_spacing / (ss_spacing + ss_residual)
eta_p_interaction = ss_interaction / (ss_interaction + ss_residual)

# Calculate Cohen's d for large vs small screen
u_large = df_factorial_pilot[df_factorial_pilot['Screen_Size'] == 'large']['Reading_Time'].mean()
u_small = df_factorial_pilot[df_factorial_pilot['Screen_Size'] == 'small']['Reading_Time'].mean()

s = sqrt(model_factorial.mse_resid)

d_screen = (u_large - u_small) / s

print(f"--- Factorial Effect Size Summary ---")
print(f"Screen Size (Partial Eta-Squared): {eta_p_screen:.3f}")
print(f"Line Spacing (Partial Eta-Squared): {eta_p_spacing:.3f}")
print(f"Interaction (Partial Eta-Squared): {eta_p_interaction:.3f}")
print(f"Large vs. Small Screen (Cohen's d): {d_screen:.3f}")

--- Factorial Effect Size Summary ---
Screen Size (Partial Eta-Squared): 0.217
Line Spacing (Partial Eta-Squared): 0.001
Interaction (Partial Eta-Squared): 0.009
Large vs. Small Screen (Cohen's d): -1.074


##Type 3: Crossover RCT

By using a Crossover design, each participant serves as their own control. This effectively "cancels out" the individual variance in reading speed and helps us to focus on understanding how specific typefaces may alter reading times within individual reader differences. We are currently using Courier, but think that Helvetica could appeal to fans of Swiss design.


We run each leg of our experiment for 3 weeks so the participants are exposed to our weekly newsletter and reading times are taken at weekly intervals over the 3 week period.


We plan to control screen size, which we have seen has a significant impact on reading time and spacing.


**Treatments:** Font face: {Helvetica, Courier}

**Order:**

*HC:* People have Helvetica for 3 weeks, with a 2 week washout, then move to Courier for 3 weeks.

*CH:* Others have Courier for 3 weeks, with a 2 week washout, then move to Helvetica for 3 weeks.

**Outcome:** Reading time

**Main effect**

**H_01:** No main effect of typeface on reading time.

**H_A1:** Typeface affects on reading time.

**Sequence effect**

**H_02:** No sequence effect.

**H_A2:** The order in which the typefaces are presented has an effect on reading time.

**Period effect**

**H_03:** No period effect.

**H_A3:** The period has an effect on time spent.

**Treatment:** Helvetica

**Control:** Courier

**Other Features:** Participants are randomly assigned to font order (HC or CH), and the experiment uses a randomized complete block design with blocks defined by reading difficulty (Easy, Medium, Hard).

**Subset Data**

In [ ]:
# Create the pilot dataset using synthetic data
crossover_pilot_raw = generate_trial_data(20)

# Filter for treatment periods (excluding washout)
df_crossover = crossover_pilot_raw[crossover_pilot_raw['Period'].isin(['Period_1', 'Period_2'])].copy()

# Add Sequence column based on Participant_ID
df_crossover['Sequence'] = df_crossover['Participant_ID'].apply(lambda x: 'CH' if x % 2 == 0 else 'HC')

# Add blocking factor to the crossover subset only
np.random.seed(42)
df_crossover['Reading_Difficulty'] = np.random.choice(
    ['easy', 'medium', 'hard'],
    size=len(df_crossover)
)

# Final analysis dataset
df_crossover_final = df_crossover.groupby(
    ['Participant_ID', 'Sequence', 'Period', 'Typeface', 'Reading_Difficulty']
)['Reading_Time'].mean().reset_index()

print("Crossover Subset Data (Aggregated for Mixed Model):")
print(df_crossover_final.head(6))

Crossover Subset Data (Aggregated for Mixed Model):
   Participant_ID Sequence    Period   Typeface Reading_Difficulty  \
0               0       CH  Period_1    courier               easy   
1               0       CH  Period_1    courier               hard   
2               0       CH  Period_2  helvetica               easy   
3               0       CH  Period_2  helvetica               hard   
4               1       HC  Period_1  helvetica               hard   
5               1       HC  Period_1  helvetica             medium   

   Reading_Time  
0       157.750  
1       156.085  
2       167.445  
3       170.830  
4       154.675  
5       154.820  


**Adding blocking factor**

In [ ]:
# Add blocking factor to the crossover subset only
np.random.seed(42)
df_crossover['Reading_Difficulty'] = np.random.choice(
    ['easy', 'medium', 'hard'],
    size=len(df_crossover)
)

**Model & Results 2-Way Mixed ANOVA (Crossover RCT)**



In [ ]:
# As RM mixed_anova doesn't support 2 within factors, we are using a multilevel
# model for this analysis

df_crossover_final = df_crossover.groupby(
    ['Participant_ID', 'Sequence', 'Period', 'Typeface', 'Reading_Difficulty']
)['Reading_Time'].mean().reset_index()

# Fit the Mixed Linear Model with blocking factor
model_mixed = smf.mixedlm(
    "Reading_Time ~ Typeface + Period + Sequence + Reading_Difficulty",
    data=df_crossover_final,
    groups=df_crossover_final["Participant_ID"]
).fit()

print("--- Crossover RCT: Mixed Linear Model Results with Blocking Factor ---")
print(model_mixed.summary())

--- Crossover RCT: Mixed Linear Model Results with Blocking Factor ---
                   Mixed Linear Model Regression Results
Model:                   MixedLM      Dependent Variable:      Reading_Time
No. Observations:        78           Method:                  REML        
No. Groups:              20           Scale:                   7.0714      
Min. group size:         2            Log-Likelihood:          -202.7352   
Max. group size:         6            Converged:               Yes         
Mean group size:         3.9                                               
---------------------------------------------------------------------------
                              Coef.  Std.Err.    z    P>|z|  [0.025  0.975]
---------------------------------------------------------------------------
Intercept                    153.243    1.514 101.194 0.000 150.275 156.211
Typeface[T.helvetica]          7.719    0.615  12.545 0.000   6.513   8.925
Period[T.Period_2]             7.876

**Effect Calculation**

In [ ]:
# One row per participant per typeface
df_effect_size = df_crossover_final.groupby(['Participant_ID', 'Typeface'])['Reading_Time'].mean().reset_index()

# Pivot
df_pivot = df_effect_size.pivot(index='Participant_ID', columns='Typeface', values='Reading_Time')

# Calculate means and SDs
mean_h = df_pivot['helvetica'].mean()
mean_c = df_pivot['courier'].mean()
sd_h = df_pivot['helvetica'].std()
sd_c = df_pivot['courier'].std()

# Formula for d_av
cohens_d_av = (mean_h - mean_c) / ((sd_h + sd_c) / 2)

print(f"--- Effect Size Calculation ---")
print(f"Helvetica Mean: {mean_h:.2f} and std {sd_h:.2f}")
print(f"Courier Mean:   {mean_c:.2f} and std {sd_c:.2f}")
print(f"Cohen's d_av:   {abs(cohens_d_av):.3f}")

--- Effect Size Calculation ---
Helvetica Mean: 160.50 and std 9.41
Courier Mean:   152.85 and std 4.53
Cohen's d_av:   1.098


In [ ]:
sequence_means = df_crossover_final.groupby(['Sequence'])['Reading_Time'] \
    .agg(['mean', 'std']) \
    .round(2)

print("--- Crossover RCT: Sequence Means with SD ---")
print(sequence_means)

--- Crossover RCT: Sequence Means with SD ---
            mean   std
Sequence              
CH        160.61  8.85
HC        152.14  5.29


In [ ]:
period_means = df_crossover_final.groupby(['Period'])['Reading_Time'] \
    .agg(['mean', 'std']) \
    .round(2)

print("--- Crossover RCT: Period Means with SD ---")
print(period_means)

--- Crossover RCT: Period Means with SD ---
            mean   std
Period                
Period_1  152.51  5.19
Period_2  160.04  9.25


In [ ]:
period_means = df_crossover_final.groupby(['Typeface'])['Reading_Time'] \
    .agg(['mean', 'std']) \
    .round(2)

print("--- Crossover RCT: Period Means with SD ---")
print(period_means)

--- Crossover RCT: Period Means with SD ---
             mean   std
Typeface               
courier    152.70  4.82
helvetica  160.44  9.64


## Type 4: Withdrawal RCT

By using a withdrawal randomized controlled trial (RCT) design, we aimed to evaluate whether reducing line spacing after an initial exposure period would affect newsletter reading time. All participants were first exposed to the newsletter with 1.75 line spacing.


After this initial period, half of the participants were randomly assigned to switch to single line spacing, while the remaining participants continued with 1.75 spacing and served as the control group. This design allowed us to observe whether removing the additional spacing after readers had already become accustomed to the newsletter would lead to measurable changes in reading time.


The experiment was conducted over eight weeks. During weeks 1–4, all participants read the newsletter with 1.75 spacing. During weeks 5–8, the treatment group switched to single spacing, while the control group maintained 1.75 spacing throughout the entire study.


To reduce potential confounding influences, screen size and typeface were randomized across participants.


**Outcome Variable:** Reading time

**Between subjects factor:** Line spacing (1.75 and single)

**Within subjects factor:** Week

**Other Features:** The participants were randomly assigned to newsletter conditions in blocks, grouped by reading difficulty.


**Main effect of line spacing**

**H_01:** No main effect of line spacing on reading time.

**H_A1:** Line spacing does affect reading time.

**Main effect of week**

**H_02:** No main effect of week on reading time.

**H_A2:** Week does affect reading time.

**Interaction effect**

**H_03:** No treatment × week interaction.

**H_A3:** There is a treatment and week interaction.


**Subset Data**

In [ ]:
withdrawal_pilot_raw = generate_trial_data(60)

df_before = withdrawal_pilot_raw[withdrawal_pilot_raw['Week_Number'] <= 4]\
            .groupby(['Participant_ID', 'Withdrawal_Group', 'Screen_Size']) \
            ['Reading_Time'].mean().reset_index()
df_before.rename(columns={'Reading_Time': 'Reading_Time_Baseline'}, inplace=True)

df_after = withdrawal_pilot_raw[withdrawal_pilot_raw['Week_Number'] == 8]\
           [['Participant_ID', 'Reading_Time']]
df_after.rename(columns={'Reading_Time': 'Reading_Time_Outcome'}, inplace=True)

df_withdrawal_wide = pd.merge(df_before, df_after, on='Participant_ID')

df_baseline = df_withdrawal_wide[['Participant_ID', 'Withdrawal_Group', 'Reading_Time_Baseline']].copy()
df_baseline['Time'] = 'Before'
df_baseline.rename(columns={'Reading_Time_Baseline': 'Reading_Time'}, inplace=True)

df_outcome = df_withdrawal_wide[['Participant_ID', 'Withdrawal_Group', 'Reading_Time_Outcome']].copy()
df_outcome['Time'] = 'After'
df_outcome.rename(columns={'Reading_Time_Outcome': 'Reading_Time'}, inplace=True)

df_mixed_final = pd.concat([df_baseline, df_outcome])
print("Withdrawal Subset Data (Baseline vs. Outcome):")
print(df_mixed_final.head(6))

Withdrawal Subset Data (Baseline vs. Outcome):
   Participant_ID     Withdrawal_Group  Reading_Time    Time
0               0  Withdraw_to_Control      157.5575  Before
1               1    Stay_on_Treatment      154.4500  Before
2               2  Withdraw_to_Control      156.4975  Before
3               3    Stay_on_Treatment      150.4675  Before
4               4  Withdraw_to_Control      147.2075  Before
5               5    Stay_on_Treatment      152.8325  Before


**Adding blocking factor**

In [ ]:
# Add blocking factor to the withdrawal subset only
np.random.seed(42)
df_withdrawal_wide['Reading_Difficulty'] = np.random.choice(
    ['easy', 'medium', 'hard'],
    size=len(df_withdrawal_wide)
)

**Model & Results Two-Way Mixed ANOVA (Withdrawal RCT)**



In [ ]:
# Fit ANCOVA with blocking factor
formula = 'Reading_Time_Outcome ~ C(Withdrawal_Group) * Reading_Time_Baseline + C(Reading_Difficulty)'

model_withdrawal = ols(formula, data=df_withdrawal_wide).fit()
anova_results = sm.stats.anova_lm(model_withdrawal, typ=2)

print("--- Withdrawal RCT ANCOVA Results with Blocking Factor ---")
print(anova_results)

--- Withdrawal RCT ANCOVA Results with Blocking Factor ---
                                                sum_sq    df           F  \
C(Withdrawal_Group)                        2921.211990   1.0  271.762615   
C(Reading_Difficulty)                         3.010823   2.0    0.140050   
Reading_Time_Baseline                      1148.719385   1.0  106.866255   
C(Withdrawal_Group):Reading_Time_Baseline    10.825486   1.0    1.007103   
Residual                                    580.453081  54.0         NaN   

                                                 PR(>F)  
C(Withdrawal_Group)                        9.950807e-23  
C(Reading_Difficulty)                      8.696299e-01  
Reading_Time_Baseline                      2.083461e-14  
C(Withdrawal_Group):Reading_Time_Baseline  3.200737e-01  
Residual                                            NaN  


In [ ]:
cell_means_full = df_mixed_final.groupby(['Withdrawal_Group', 'Time'])['Reading_Time'] \
    .agg(['mean', 'std']) \
    .round(2)

print("--- Withdrawl RCT: Cell Means with SD ---")
print(cell_means_full)

--- Withdrawl RCT: Cell Means with SD ---
                              mean   std
Withdrawal_Group    Time                
Stay_on_Treatment   After   151.40  6.04
                    Before  151.75  4.79
Withdraw_to_Control After   164.64  5.60
                    Before  150.80  5.88


**Effect Calculation**

In [ ]:
# Effect size calculation: partial eta-squared
ss_group = anova_results.loc['C(Withdrawal_Group)', 'sum_sq']
ss_residual = anova_results.loc['Residual', 'sum_sq']

eta_p_withdrawal = ss_group / (ss_group + ss_residual)

print("--- Withdrawal Effect Size Summary ---")

print(f"Withdrawal Group (Partial Eta-Squared): {eta_p_withdrawal:.3f}")

--- Withdrawal Effect Size Summary ---
Withdrawal Group (Partial Eta-Squared): 0.834


## Type 5: Matched pairs RCT

Our objective is to further understand the impact of larger line spacing on reading times. We expect that vision degrades with age, so we plan to match our participants according to age.

Our general experimental setup was as follows:
Paired two different people who are similar in age.
One person in the pair got the treatment, the other served as the control.

We controlled for screen size (large) and font (Courier).

**Factor:** Line spacing (1.75 and single)

**Control:** 1pt spacing

**Treatment:** 1.75 spacing

**Matching factor:** Age

**Outcome variable:** Reading_Time

**Other features:** The participants were randomly assigned to newsletter conditions in blocks, grouped by reading difficulty.

**H_0:** The mean difference in Reading_Time between the treated individual (1.75 spacing) and their matched control individual (single spacing) is zero.

**H_A1:** There is a significant difference in Reading_Time between the treated individuals and their matched controls.

**Subset Data**

In [ ]:
# Create the raw dataset using synthetic data
# We generate enough participants to ensure a healthy match pool
matched_rct_raw = generate_trial_data(34)

# Filter for Year 2 and the target spacings
# We isolate the specific Treatment (1.75) and Control (Single)
df_matched_base = matched_rct_raw[
    (matched_rct_raw['Time_Point'] == 'Week_8') &
    (matched_rct_raw['Line_Spacing'].isin(['single', 'one_three_quarters']))
].copy()

# Separate and Sort for Age-Matching
# This ensures Pair 1 is the two youngest, Pair 2 the next, and so on
group_control = df_matched_base[df_matched_base['Line_Spacing'] == 'single'].sort_values('Age')
group_treatment = df_matched_base[df_matched_base['Line_Spacing'] == 'one_three_quarters'].sort_values('Age')

# Create Matched Pairs (Pairing the closest age matches)
# We find the smallest group size to ensure every participant has a match
min_size = min(len(group_control), len(group_treatment))
df_matched_final = pd.DataFrame({
    'Pair_ID': range(min_size),
    'Age_Control': group_control['Age'].iloc[:min_size].values,
    'Age_Treatment': group_treatment['Age'].iloc[:min_size].values,
    'Time_Control': group_control['Reading_Time'].iloc[:min_size].values,
    'Time_Treatment': group_treatment['Reading_Time'].iloc[:min_size].values
})

# Calculate Age Difference for validation
df_matched_final['Age_Diff'] = abs(df_matched_final['Age_Control'] - df_matched_final['Age_Treatment'])

# Preview the matched pairs
print("Matched Pairs Subset (Age-Matched Control vs Treatment):")
print(df_matched_final[['Pair_ID', 'Age_Control', 'Age_Treatment', 'Age_Diff', 'Time_Control', 'Time_Treatment']].head(6))

Matched Pairs Subset (Age-Matched Control vs Treatment):
   Pair_ID  Age_Control  Age_Treatment  Age_Diff  Time_Control  Time_Treatment
0        0           26             19         7        146.54          157.98
1        1           29             27         2        148.93          166.99
2        2           32             30         2        168.19          153.97
3        3           40             36         4        154.92          154.82
4        4           49             37        12        157.03          153.18
5        5           49             44         5        168.35          167.96


**Adding blocking factor**

In [ ]:
# Add blocking factor to the matched-pairs subset only
np.random.seed(42)
df_matched_final['Reading_Difficulty'] = np.random.choice(
    ['easy', 'medium', 'hard'],
    size=len(df_matched_final)
)

**Model & Results Repeated Measures ANOVA (Matched pairs: RCT)**

In [ ]:
# Melt the data into long format
df_matched_long = df_matched_final.melt(
    id_vars=['Pair_ID', 'Reading_Difficulty'],
    value_vars=['Time_Control', 'Time_Treatment'],
    var_name='Line_Spacing',
    value_name='Reading_Time'
)

# Rename levels for clarity
df_matched_long['Line_Spacing'] = df_matched_long['Line_Spacing'].replace({
    'Time_Control': 'single',
    'Time_Treatment': 'one_three_quarters'
})

# OLS blocking model with Pair_ID and Reading_Difficulty
formula = 'Reading_Time ~ C(Line_Spacing) + C(Pair_ID) + C(Reading_Difficulty)'

model_matched = ols(formula, data=df_matched_long).fit()
anova_matched_results = sm.stats.anova_lm(model_matched, typ=2)

print("\n--- MATCHED PAIRS ANOVA RESULTS WITH BLOCKING FACTOR ---")
print(anova_matched_results.round(4))


--- MATCHED PAIRS ANOVA RESULTS WITH BLOCKING FACTOR ---
                           sum_sq    df        F  PR(>F)
C(Line_Spacing)           18.8793   1.0   0.3237  0.5819
C(Pair_ID)             43959.6500  10.0  75.3826  0.0000
C(Reading_Difficulty)   4351.8189   2.0  37.3128  0.0000
Residual                 583.1536  10.0      NaN     NaN


In [ ]:
cell_means_matched = df_matched_long.groupby(['Line_Spacing'])['Reading_Time'] \
    .agg(['mean', 'std']) \
    .round(2)

print("--- Paired RCT: Cell Means with SD ---")
print(cell_means_matched)

--- Paired RCT: Cell Means with SD ---
                      mean   std
Line_Spacing                    
one_three_quarters  159.08  8.51
single              160.93  8.60


**Effect Calculation**

In [ ]:
# Calculate the raw difference for each pair
df_matched_final['diff'] = df_matched_final['Time_Control'] - df_matched_final['Time_Treatment']

mean_diff = df_matched_final['diff'].mean()
std_diff = df_matched_final['diff'].std()
cohens_dz = mean_diff / std_diff

# Extract ANOVA-based effect sizes
f_val = anova_matched_results.loc['C(Line_Spacing)', 'F']
df_effect = anova_matched_results.loc['C(Line_Spacing)', 'df']
df_error = anova_matched_results.loc['Residual', 'df']

r_effect = np.sqrt((f_val * df_effect) / (f_val * df_effect + df_error))

ss_spacing = anova_matched_results.loc['C(Line_Spacing)', 'sum_sq']
ss_residual = anova_matched_results.loc['Residual', 'sum_sq']
eta_p_spacing = ss_spacing / (ss_spacing + ss_residual)

print(f"--- Effect Size Metrics ---")
print(f"Mean Difference:              {mean_diff:.2f} seconds")
print(f"Cohen's dz:                   {abs(cohens_dz):.3f}")
print(f"Correlation (r):              {r_effect:.3f}")
print(f"Partial Eta-Squared:          {eta_p_spacing:.3f}")

--- Effect Size Metrics ---
Mean Difference:              1.85 seconds
Cohen's dz:                   0.172
Correlation (r):              0.177
Partial Eta-Squared:          0.031
